In [10]:
# =============================================
# PART B1: PYTHON - DATA CLEANING & PREPARATION
# =============================================

# 1. Loading & Exploring Dataset
import pandas as pd
import numpy as np

# Load the raw dataset using pd.read_csv()
df = pd.read_csv('startup_funding.csv')

# Display the first 5 rows to understand the structure
print("\n--- FIRST 5 ROWS ---")
print(df.head(5))

# Check and print missing values for every single column explicitly
print("\n--- MISSING VALUE COUNT PER COLUMN ---")
print(df.isnull().sum())


# 2. Rename Columns Necessary for Consistency
df.rename(columns={'City  Location': 'City Location', 'InvestmentnType': 'Investment Type', 'SubVertical': 'Sub Vertical'}, inplace=True)


# 3. Removing Duplicates
# Count rows before removing duplicates
before_dedup = len(df)
# Drop completely duplicate records from the dataset
df.drop_duplicates(inplace=True)
print("\nDropped",before_dedup,"-",len(df),"duplicate rows.")


# 4. Missing Value Handling with Comments
# Categorical columns must be filled with standard placeholders so null values do not break later aggregations or visualizations.
# We fill structural text columns with 'Unknown' and open-ended commentary fields with 'No Remarks'.
df['Industry Vertical'] = df['Industry Vertical'].fillna('Unknown')
df['Sub Vertical'] = df['Sub Vertical'].fillna('Unknown')
df['City Location'] = df['City Location'].fillna('Unknown')
df['Investors Name'] = df['Investors Name'].fillna('Unknown')
df['Investment Type'] = df['Investment Type'].fillna('Unknown')
df['Remarks'] = df['Remarks'].fillna('No Remarks')

# Isolate missing financial values as string '0' prior to cleaning data types
df['Amount in USD'] = df['Amount in USD'].fillna('0')


# 5. Cleaning Text & Formatting Data Types
# Apply .strip() to eliminate hidden padding spaces and convert text to .title() for uniform visual analytics
text_cols = ['Startup Name', 'Industry Vertical', 'Sub Vertical', 'City Location', 'Investors Name', 'Investment Type']
for col in text_cols:
  df[col] = df[col].astype(str).str.strip().str.title()

# Clean up 'Remarks' text by removing leading/trailing spaces
df['Remarks'] = df['Remarks'].astype(str).str.strip()

# Fix incorrect data types for 'Amount in USD' column:
# 1. Convert to string and remove formatting commas
df['Amount in USD'] = df['Amount in USD'].astype(str).str.replace(',', '').str.strip()
# 2. Replace text anomalies like 'N/A' or 'Undisclosed' with string '0' so it can parse to numeric integers
df['Amount in USD'] = df['Amount in USD'].replace(['N/A', 'nan', 'N/A', ''], '0')
# 3. Force format conversion to full numeric integers, filling unexpected errors with 0
df['Amount in USD'] = pd.to_numeric(df['Amount in USD'], errors='coerce').fillna(0).astype(int)

# Extract a numeric 'Year' column from 'Date dd/mm/yyyy'
df['Year'] = df['Date dd/mm/yyyy'].astype(str).str.extract(r'()\d{4}')
df['Year'] = pd.to_numeric(df['Year'], errors='coerce').fillna(2018).astype(int)

# Clean up column names (lowercase with single underscores) for simpler SQL typing
df.columns = df.columns.str.strip().str.replace(r'\s+', '_', regex=True).str.replace('/', '_').str.lower()


# 6. Creating at Least One New Useful Column
# Segment startup investments into tiers based on capital amount size in USD
def classify_funding_scale(amount):
  if amount == 0: return 'Undisclosed'
  elif amount < 2000000: return 'Seed & Early Stage'
  elif amount < 50000000: return 'Growth Capital'
  else: return 'Late Stage Mega Round'

df['funding_tier'] = df['amount_in_usd'].apply(classify_funding_scale)


# 7. Exporting Clean Dataset + Comments Quality
print("\n--- FINAL CLEANED DATA SUMMARY ---")
print(df.info())

# Save the pristine file as cleaned_dataset.csv as explicitly requested
df.to_csv('cleaned_dataset.csv', index=False)
print("\nSaved successfully as 'cleaned_dataset.csv'")


--- FIRST 5 ROWS ---
   Sr No Date dd/mm/yyyy                  Startup Name    Industry Vertical  \
0      1      09/01/2020                        BYJU’S               E-Tech   
1      2      13/01/2020                        Shuttl       Transportation   
2      3      09/01/2020                     Mamaearth           E-commerce   
3      4      02/01/2020  https://www.wealthbucket.in/              FinTech   
4      5      02/01/2020                        Fashor  Fashion and Apparel   

                             SubVertical City  Location  \
0                             E-learning      Bengaluru   
1              App based shuttle service        Gurgaon   
2  Retailer of baby and toddler products      Bengaluru   
3                      Online Investment      New Delhi   
4            Embroiled Clothes For Women         Mumbai   

              Investors Name       InvestmentnType Amount in USD Remarks  
0    Tiger Global Management  Private Equity Round  20,00,00,000     NaN 